# 09 Epoch AI models table (compute, cost, accessibility)

**Purpose.** This notebook reads Epoch AI's table of AI models (`all_ai_models.csv`) and keeps the columns that matter for the project: the developer, its country, when the model was published, how it can be accessed, and how much compute and money went into training it. The table gives compute and cost context for Q2 and adds models that the ECI table does not list.

**Question served.** Q2 (compute and cost context). Optional source, ranked below the ECI table.

**How to run.** Run all cells from top to bottom. The notebook tries to download the file. If the download is blocked, it uses the copy you saved by hand in `Data/raw/epoch/`. Epoch's server answered 403 to the script that we used for testing, so expect to save this file by hand the first time.

## At a glance: links, access, download, timing

| Item | Detail |
|---|---|
| Kind | Download of one CSV |
| Publisher | Epoch AI |
| Data page | https://epoch.ai/data/ai-models |
| File used | https://epoch.ai/data/all_ai_models.csv |
| Access | Public file. No account, key or login needed. No scraping |
| Download size | not recorded. The file has about 57 columns and is updated daily |
| Expected time | download 2 to 10 seconds, whole notebook under 20 seconds |
| Data as of | the extraction day, and the latest publication date inside the file. See `processed/data_currency.csv` |
| To update | Run the notebook again. If the download is blocked, save the file by hand, see the update section |
| Tested with | A small made-up table with the expected column names, because the real file could not be downloaded from the test machine. **The first run on a real file is the real test.** The notebook prints which expected columns it found |

## 1. Source and reference links

| Item | Link |
|---|---|
| Data page for AI models | https://epoch.ai/data/ai-models |
| File | https://epoch.ai/data/all_ai_models.csv |
| All Epoch datasets | https://epoch.ai/data |
| Epoch documentation of the table | the documentation link on the data page above |

**Citation.** Epoch AI, 'Data on AI Models'. Published online at epoch.ai. Retrieved from https://epoch.ai/data/ai-models [online resource]. Use the citation block shown on the data page if it differs.

## 2. Table structure

### 2.1 Source file

One row per AI model. About 57 columns. The ones we use are listed below. Column names are matched without regard to capital letters or extra spaces, so a small change in wording does not break the run. A missing optional column is reported and skipped.

| Source column (as expected) | Our column | Needed? |
|---|---|---|
| `Model` | `model` | yes |
| `Organization` | `organization` | yes |
| `Country (of organization)` | `country` | yes |
| `Publication date` | `publication_date` | yes |
| `Model accessibility` | `model_accessibility` | no |
| `Domain` | `domain` | no |
| `Parameters` | `parameters` | no |
| `Training compute (FLOP)` | `training_compute_flop` | no |
| `Training compute cost (2023 USD)` | `training_compute_cost_usd_2023` | no |
| `Confidence` | `confidence` | no |
| `Organization categorization` | `organization_type` | no |
| `Link` | `source_link` | no |

### 2.2 Outputs in `processed/`

| File | Content |
|---|---|
| `ai_models.csv` | One row per model with the columns above, plus `is_china_based` (country contains China or Hong Kong) and `has_compute` (training compute is stated) |
| `ai_models_columns.csv` | Every column of the source file with the share of rows that are filled in. This lets you see what else the file offers without opening it |
| `data_currency.csv` | When extracted, what period the data covers |

## 3. Set up the paths and download helper

In [1]:
import hashlib
import json
import shutil
import time
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
import requests

T0 = time.time()
FOLDER = "09_epoch_ai_models"
cwd = Path.cwd()
if cwd.name == FOLDER:
    HERE = cwd
elif (cwd / FOLDER).exists():
    HERE = cwd / FOLDER
elif (cwd / "Data" / FOLDER).exists():
    HERE = cwd / "Data" / FOLDER
else:
    raise FileNotFoundError(f"Open this notebook from the folder {FOLDER} (or from Data/). Current folder: {cwd}")

DATA = HERE.parent               # the Data folder
MANUAL = DATA / "raw" / "epoch"                # hand-downloaded or pre-copied source files
RAW = HERE / "raw"               # files this notebook downloads, kept exactly as received
OUT = HERE / "processed"         # tables this notebook writes
for folder in (RAW, OUT):
    folder.mkdir(parents=True, exist_ok=True)
LOG_PATH = RAW / "retrieval_log.json"
print("Notebook folder:", HERE)
print("Manual folder  :", MANUAL)


HEADERS = {"User-Agent": "UTS-36104-A3-student-project (academic use, no contact data stored)"}
TIMEOUT = 120
REFRESH = True   # True: try a fresh download first. False: reuse the file already in raw/ if there is one.


def record(source_id: str, url: str, path: Path, how: str) -> None:
    """Add (or replace) the retrieval-log entry for one raw file."""
    content = path.read_bytes()
    log = json.loads(LOG_PATH.read_text(encoding="utf-8")) if LOG_PATH.exists() else {}
    log[source_id] = {
        "url": url,
        "file": path.name,
        "how": how,
        "retrieved_utc": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
        "bytes": len(content),
        "sha256": hashlib.sha256(content).hexdigest(),
    }
    LOG_PATH.write_text(json.dumps(log, indent=2), encoding="utf-8")
    print(f"logged {source_id}: {len(content):,} bytes via {how}")


def get_file(source_id: str, url: str, target: str, manual_name: str = None) -> Path:
    """Return the path of a raw file. Order: fresh download, earlier download in raw/, manual copy."""
    path = RAW / target
    if REFRESH or not path.exists():
        try:
            response = requests.get(url, headers=HEADERS, timeout=TIMEOUT)
            response.raise_for_status()
            path.write_bytes(response.content)
            record(source_id, url, path, "downloaded")
            return path
        except Exception as err:
            print(f"Download of {source_id} failed: {err}")
    manual = MANUAL / (manual_name or target)
    candidates = [p for p in (path, manual) if p.exists()]
    if candidates:
        best = max(candidates, key=lambda p: p.stat().st_mtime)   # the newest copy wins
        if best == path:
            print(f"Using the earlier download {path.name}")
        else:
            shutil.copy2(best, path)
            record(source_id, url, path, f"copied from manual folder {manual.parent.name}")
        return path
    raise FileNotFoundError(
        f"Could not get {source_id}. Open {url} in a browser, save the file as '{manual_name or target}' "
        f"inside {MANUAL}, then run this cell again.")

import re

Notebook folder: d:\UTS MDSI\36104 (2026)\Assignment 3\Hedging Region AI Model Selection and Data Risk\Data\09_epoch_ai_models
Manual folder  : d:\UTS MDSI\36104 (2026)\Assignment 3\Hedging Region AI Model Selection and Data Risk\Data\raw\epoch


## 4. Acquire the file

In [2]:
MODELS_URL = "https://epoch.ai/data/all_ai_models.csv"
csv_path = get_file("all_ai_models", MODELS_URL, "all_ai_models.csv")
raw = pd.read_csv(csv_path, low_memory=False)
print(raw.shape)
print("First columns:", list(raw.columns[:8]))

logged all_ai_models: 6,827,261 bytes via downloaded
(3627, 57)
First columns: ['Model', 'Domain', 'Task', 'Organization', 'Authors', 'Publication date', 'Reference', 'Link']


## 5. Inspect the columns

In [3]:
def norm(name):
    return re.sub(r"\s+", " ", str(name)).strip().lower()

fill = pd.DataFrame({"column": raw.columns,
                     "share_filled": [round(float(raw[c].notna().mean()), 3) for c in raw.columns],
                     "example": [str(raw[c].dropna().iloc[0])[:60] if raw[c].notna().any() else "" for c in raw.columns]})
fill.to_csv(OUT / "ai_models_columns.csv", index=False, encoding="utf-8")
print(len(fill), "columns written to ai_models_columns.csv")
fill.head(15)

57 columns written to ai_models_columns.csv


,column,share_filled,example
0,Model,1.000,Muse Spark Open
1,Domain,0.980,"Language,Multimodal,Vision,Audio,Video"
2,Task,0.972,"Language modeling/generation,Question answering"
3,Organization,0.979,Meta AI
4,Authors,0.743,"Xin Cheng, Xin Liu, Xin Xie, Xinbo Xu, Xingcha..."
5,Publication date,0.997,2026-10-15
6,Reference,0.954,Gemini 4 Argon: our next era of frontier intel...
7,Link,0.989,https://research.meta.ai/blog/muse-spark-open\...
8,Citations,0.409,308.0
9,Notability criteria,0.304,Discretionary


## 6. Keep the project columns

The wanted columns are matched by normalised name. The notebook stops only if one of the four required columns is missing.

In [4]:
WANTED = [   # (source name, our name, required)
    ("Model", "model", True),
    ("Organization", "organization", True),
    ("Country (of organization)", "country", True),
    ("Publication date", "publication_date", True),
    ("Model accessibility", "model_accessibility", False),
    ("Domain", "domain", False),
    ("Parameters", "parameters", False),
    ("Training compute (FLOP)", "training_compute_flop", False),
    ("Training compute cost (2023 USD)", "training_compute_cost_usd_2023", False),
    ("Confidence", "confidence", False),
    ("Organization categorization", "organization_type", False),
    ("Link", "source_link", False),
]
lookup = {norm(c): c for c in raw.columns}
picked, missing_required, missing_optional = {}, [], []
for source, ours, required in WANTED:
    if norm(source) in lookup:
        picked[lookup[norm(source)]] = ours
    else:
        (missing_required if required else missing_optional).append(source)
print("Found   :", list(picked.values()))
print("Missing optional (skipped):", missing_optional if missing_optional else "none")
if missing_required:
    raise KeyError(f"Required columns not found: {missing_required}. Open ai_models_columns.csv in processed/ to see the real names, then edit WANTED above.")

models = raw[list(picked)].rename(columns=picked).copy()
models["publication_date"] = pd.to_datetime(models["publication_date"], errors="coerce")
for col in ("parameters", "training_compute_flop", "training_compute_cost_usd_2023"):
    if col in models.columns:
        models[col] = pd.to_numeric(models[col], errors="coerce")

def china_flag(value):
    """True when any listed country is China or Hong Kong. The field can hold several countries."""
    if pd.isna(value):
        return False
    parts = [p.strip().lower() for p in re.split(r"[,;/]", str(value))]
    return any(p in ("china", "hong kong", "people's republic of china") for p in parts)

models["is_china_based"] = models["country"].map(china_flag)
models["has_compute"] = models["training_compute_flop"].notna() if "training_compute_flop" in models.columns else False
models = models.sort_values("publication_date", ascending=False).reset_index(drop=True)
print(models.shape)
models.head(8)

Found   : ['model', 'organization', 'country', 'publication_date', 'model_accessibility', 'domain', 'parameters', 'training_compute_flop', 'training_compute_cost_usd_2023', 'confidence', 'organization_type', 'source_link']
Missing optional (skipped): none
(3627, 14)


,model,organization,country,publication_date,model_accessibility,domain,parameters,training_compute_flop,training_compute_cost_usd_2023,confidence,organization_type,source_link,is_china_based,has_compute
0,Muse Spark Open,Meta AI,United States of America,2026-10-15,Open weights (restricted use),"Language,Multimodal,Vision,Audio,Video",NaN,NaN,NaN,Confident,Industry,https://research.meta.ai/blog/muse-spark-open\...,False,False
1,Gemini 4 Argon,Google DeepMind,United States of America,2026-09-30,Limited access,"Language,Multimodal,Vision",NaN,NaN,NaN,Confident,Industry,https://blog.google/innovation-and-ai/models-a...,False,False
2,GPT-6.1 Sol,OpenAI,United States of America,2026-09-29,API access,"Multimodal,Language,Vision",NaN,NaN,NaN,Confident,Industry,https://openai.com/index/introducing-gpt-6-1-sol/,False,False
3,Claude Sonnet 5.5,Anthropic,United States of America,2026-09-28,API access,"Language,Multimodal,Vision",NaN,NaN,NaN,Confident,Industry,https://www.anthropic.com/claude-sonnet-5-5,False,False
4,Claude Opus 5.5,Anthropic,United States of America,2026-09-22,API access,Language,NaN,NaN,NaN,Confident,Industry,https://www.anthropic.com/claude-opus-5-5,False,False
5,GPT-6 Sol,OpenAI,United States of America,2026-09-22,API access,"Multimodal,Language,Vision",NaN,NaN,NaN,Confident,Industry,https://openai.com/index/introducing-gpt-6-sol...,False,False
6,GPT-6 Luna,OpenAI,United States of America,2026-09-22,API access,"Multimodal,Language,Vision",NaN,NaN,NaN,Confident,Industry,https://openai.com/index/introducing-gpt-6-sol...,False,False
7,Grok 4.7,xAI,United States of America,2026-09-21,API access,Language,2.100000e+09,NaN,NaN,Likely,Industry,https://x.ai/news/grok-4-7,False,False


## 7. Validate

In [5]:
problems = []
if models["model"].isna().mean() > 0.01:
    problems.append("many rows without a model name")
if models["publication_date"].notna().mean() < 0.5:
    problems.append("fewer than half of the publication dates could be read")
if len(models) < 100:
    problems.append("fewer than 100 models, the file may be truncated")
if models["country"].notna().mean() < 0.5:
    problems.append("fewer than half of the rows have a country")
for col in ("training_compute_flop", "training_compute_cost_usd_2023", "parameters"):
    if col in models.columns and (models[col].dropna() < 0).any():
        problems.append(f"negative values in {col}")
print("Problems:", problems if problems else "none")
assert not problems, problems
print()
print("Models:", len(models), " with a stated training compute:", int(models["has_compute"].sum()),
      " China-based rows:", int(models["is_china_based"].sum()))

Problems: none

Models: 3627  with a stated training compute: 1410  China-based rows: 964


## 8. Save

In [6]:
models.to_csv(OUT / "ai_models.csv", index=False, encoding="utf-8")
print("saved", OUT / "ai_models.csv", models.shape)

saved d:\UTS MDSI\36104 (2026)\Assignment 3\Hedging Region AI Model Selection and Data Risk\Data\09_epoch_ai_models\processed\ai_models.csv (3627, 14)


## 9. Quick look: models per country and how they can be accessed

In [7]:
top = models["country"].value_counts().head(8)
print(top)
if "model_accessibility" in models.columns:
    print()
    print(pd.crosstab(models["is_china_based"], models["model_accessibility"].fillna("not stated")))

country
United States of America                                                      1233
China                                                                          604
United States of America,United States of America                              218
United Kingdom                                                                 131
China,China                                                                     71
Canada                                                                          67
South Korea                                                                     65
United States of America,United States of America,United States of America      61
Name: count, dtype: int64

model_accessibility  API access  Hosted access (no API)  Limited access  \
is_china_based                                                            
False                       334                      95               3   
True                        102                      38               0   



## How to update, and what to do if the download is blocked

**Automatic.** Run all cells again. Section 4 downloads https://epoch.ai/data/all_ai_models.csv and replaces `raw/all_ai_models.csv`. About 5 seconds.

**Manual download (use this if you see `Download of all_ai_models failed`).**

1. Open https://epoch.ai/data/ai-models in a browser. Use the download button on that page, or paste https://epoch.ai/data/all_ai_models.csv into the address bar so the browser saves the CSV. The PNG or SVG button under a chart only saves a picture.
2. If the page only offers a ZIP, unpack it and take the file named `all_ai_models.csv`.
3. Save it as `Data\raw\epoch\all_ai_models.csv`, replacing the old copy. Keep the name.
4. Run the notebook again. With no fresh download, it uses the **newer** of `raw/all_ai_models.csv` and the manual copy.

**Update check.** Open `raw/retrieval_log.json` and read the `how` field (`downloaded` or `copied from manual folder`). Then open `processed/data_currency.csv` and confirm `data_covers_to` is what you expect.

**If a column name changes.** The run stops with a message that names the missing required column. Open `processed/ai_models_columns.csv` to see the real names, then edit the `WANTED` list in Section 6.

## 10. Run time and limits

**Time to extract.** One request for a CSV with about 57 columns. Expect 2 to 10 seconds for the download and under 3 seconds for the processing, so a full run takes **under 20 seconds**. The last cell prints the measured time.

**Limits and known risks.**

1. The file changes daily. Every download is a different snapshot. The retrieval log in `raw/` records the time and hash.
2. Training compute and cost are estimates and many rows are empty. Use the `confidence` column and the `has_compute` flag before you quote a number.
3. The ECI table and the benchmark archive already carry country, accessibility and most compute figures. This table mainly adds cost and models outside the ECI list.
4. Model names differ from the ECI table, LiteLLM and LMArena. A crosswalk is a separate step.
5. The country is the country of the organisation and says nothing about where a model can be used.
6. This notebook was written before a real file was available on the test machine. The first real run decides whether the column list needs a small edit.

## 11. References

1. Epoch AI, Data on AI Models. https://epoch.ai/data/ai-models
2. Epoch AI, file. https://epoch.ai/data/all_ai_models.csv
3. Epoch AI, all datasets. https://epoch.ai/data

## Data currency: how up to date is this data?

Every notebook writes `processed/data_currency.csv` with the day it was extracted, the dates the data itself covers, and what we know about when the source started and how often it changes. Use these values for the "data as of" label in the app.

In [8]:
def write_currency(dataset, covers_from, covers_to, snapshot_note, origin_note, frequency):
    """Record how current the data is. extracted_utc is the moment this notebook ran."""
    row = {
        "dataset": dataset,
        "extracted_utc": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
        "data_covers_from": str(covers_from)[:10] if covers_from is not None else "not stated in the data",
        "data_covers_to": str(covers_to)[:10] if covers_to is not None else "not stated in the data",
        "snapshot_note": snapshot_note,
        "source_origin_note": origin_note,
        "source_update_frequency": frequency,
    }
    pd.DataFrame([row]).to_csv(OUT / "data_currency.csv", index=False, encoding="utf-8")
    for key, value in row.items():
        print(f"{key:24s} {value}")


write_currency("Epoch AI models table (ai_models.csv)",
               models["publication_date"].min(), models["publication_date"].max(),
               "live file downloaded in this run, so it is as current as the extraction day. The dates covered are the publication dates of the models in the table",
               "Start of the table not recorded in our notes. Check the data page.",
               "Updated daily according to the data page")

dataset                  Epoch AI models table (ai_models.csv)
extracted_utc            2026-10-17T22:51:45Z
data_covers_from         1950-07-02
data_covers_to           2026-10-15
snapshot_note            live file downloaded in this run, so it is as current as the extraction day. The dates covered are the publication dates of the models in the table
source_origin_note       Start of the table not recorded in our notes. Check the data page.
source_update_frequency  Updated daily according to the data page


In [9]:
elapsed = time.time() - T0
print(f"Elapsed time of this run: {elapsed:.1f} seconds")
print("Files written:")
for p in sorted(OUT.glob("*")):
    print(f"  {p.relative_to(HERE)}  ({p.stat().st_size:,} bytes)")

Elapsed time of this run: 2.4 seconds
Files written:
  processed\ai_models.csv  (803,272 bytes)
  processed\ai_models_columns.csv  (2,930 bytes)
  processed\data_currency.csv  (452 bytes)
